<a href="https://colab.research.google.com/github/sabanasersaleem-oss/QSmartGrid-Quantum-AI/blob/main/Ieee_(%D8%A7%D9%84%D8%A8%D8%AA%D8%B1%D8%A7%D8%A1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import math, random
from datetime import datetime, timedelta
import numpy as np, pandas as pd, streamlit as st

# ---------- إعداد الواجهة ----------
st.set_page_config(page_title="Q-SmartGrid", page_icon="⚡", layout="wide")
st.sidebar.title("⚙️ Controls / الإعدادات")

days = st.sidebar.slider("Days to simulate (الأيام)", 3, 14, 7)
seed = st.sidebar.number_input("Random seed", value=42, step=1)
peak_reduction_goal = st.sidebar.slider("Target peak reduction %", 0, 30, 12)
co2_factor = st.sidebar.number_input("CO₂ factor (kg/kWh)", value=0.45)
price_per_kwh = st.sidebar.number_input("Price per kWh ($)", value=0.12)
num_buildings = st.sidebar.slider("Number of buildings", 3, 12, 6)
min_load = st.sidebar.number_input("Min base load (kW)", value=50)
max_load = st.sidebar.number_input("Max base load (kW)", value=250)
random.seed(seed); np.random.seed(seed)

# ---------- توليد بيانات محاكاة ----------
def simulate_baseline(days, num_buildings, min_load, max_load):
    start = datetime.now().replace(minute=0, second=0, microsecond=0)
    ts = [start + timedelta(hours=h) for h in range(days*24)]
    recs = []
    for b in range(num_buildings):
        base = np.random.uniform(min_load, max_load)
        daily = 1.0 + 0.35*np.sin(np.linspace(0, 2*math.pi*days, days*24) + np.random.rand()*2*math.pi)
        noise = np.random.normal(0, 0.07, size=days*24)
        loads = np.clip(base * daily * (1+noise), 10, None)
        for h, t in enumerate(ts):
            if 8 <= t.hour < 18:
                loads[h] *= np.random.uniform(1.05, 1.25)
        for h, t in enumerate(ts):
            recs.append({"timestamp": ts[h], "building": f"B{b+1}", "load_kw": float(loads[h])})
    df = pd.DataFrame(recs)
    total = df.groupby("timestamp", as_index=False)["load_kw"].sum().rename(columns={"load_kw": "total_kw"})
    return df, total

def moving_average(series, window=6):
    return series.rolling(window=window, min_periods=1).mean()

df_long, df_total = simulate_baseline(days, num_buildings, min_load, max_load)
df_total["forecast_kw"] = moving_average(df_total["total_kw"])

# ---------- تحسين مبسّط ----------
win = min(len(df_total), 24)
recent = df_total.tail(win).copy()
reduct = peak_reduction_goal / 100.0
recent["optimized_kw"] = recent["forecast_kw"].values * (1 - reduct)

baseline_peak = float(np.max(recent["forecast_kw"]))
optimized_peak = float(np.max(recent["optimized_kw"]))
peak_drop_pct = (baseline_peak - optimized_peak) / baseline_peak * 100.0
energy_saved_kwh = float(np.sum(recent["forecast_kw"] - recent["optimized_kw"]))
co2_avoided = energy_saved_kwh * co2_factor
cost_saved = energy_saved_kwh * price_per_kwh

# ---------- توصيات بسيطة ----------
def gen_recs(peak_drop_pct, energy_saved_kwh, co2_avoided):
    en = []
    en.append("Shift HVAC +1.0°C during 12–17h." if peak_drop_pct>=10 else "Try +0.5°C HVAC setback at midday.")
    en.append("Schedule batch loads after 20:00." if energy_saved_kwh>200 else "Defer small loads to evening.")
    en.append("Publish a CO₂ dashboard." if co2_avoided>50 else "Pilot 1–2 buildings, then scale.")
    ar = ["اضبطي التكييف +0.5–1°C وقت الذروة.","جدوّلي الأحمال بعد 20:00.","اعملي لوحة مؤشرات للانبعاثات."]
    return en, ar

recs_en, recs_ar = gen_recs(peak_drop_pct, energy_saved_kwh, co2_avoided)

# ---------- عرض النتائج ----------
st.title("⚡ Q-SmartGrid — Generative-AI Energy Optimizer")
c1, c2, c3, c4 = st.columns(4)
c1.metric("Baseline Peak (kW)\nذروة الاستهلاك", f"{baseline_peak:.1f}")
c2.metric("Peak Reduction %\nخفض الذروة", f"{peak_drop_pct:.1f}%")
c3.metric("Energy Saved (kWh)\nالطاقة الموفّرة", f"{energy_saved_kwh:.1f}")
c4.metric("Est. Cost Saved ($)\nتوفير تقديري", f"{cost_saved:.2f}")

st.markdown("---")
st.subheader("📈 Forecast vs Optimized • التنبؤ مقابل التحسين")
st.line_chart(recent.set_index("timestamp")[["forecast_kw", "optimized_kw"]])
st.markdown("---")

st.subheader("🧠 AI Recommendations • توصيات الذكاء الاصطناعي")
t1, t2 = st.tabs(["English", "العربية"])
with t1:
    for i, r in enumerate(recs_en, 1): st.write(f"{i}. {r}")
with t2:
    for i, r in enumerate(recs_ar, 1): st.write(f"{i}. {r}")

st.markdown("---")
st.subheader("🌱 Impact • الأثر")
st.write(f"Energy saved: **{energy_saved_kwh:.1f} kWh**, CO₂ avoided: **{co2_avoided:.1f} kg**, Peak drop: **{peak_drop_pct:.1f}%**")
st.info("Tip: استخدمي أشرطة التحكم في الشريط الجانبي لمحاكاة سيناريوهات مختلفة.")


2025-11-13 11:45:14.568 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 11:45:14.570 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 11:45:14.698 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-11-13 11:45:14.699 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 11:45:14.700 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 11:45:14.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 11:45:14.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

streamlit
numpy
pandas

